# 用 MLP 做零售多步需求预测

本 Notebook 对商超、红酒、庭院和奢侈品手袋分别选择 MLP 配置。参数选择只使用最终测试期之前的四个滚动窗口；最终 28 天测试集仅评估一次。未来销量和模拟诊断真值不进入模型输入。

## 1. 数据、窗口与候选参数

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os, sys, hashlib


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from retail_forecast.common import *
from retail_forecast.neural import *

In [ ]:
MODEL_NAME = "MLP"
display(pd.DataFrame(NEURAL_CANDIDATES[MODEL_NAME]))
display(
    pd.DataFrame(
        [
            {
                "scenario": name,
                "days": len(frame),
                "zero_rate": float(frame.loc[frame.is_open, "y"].eq(0).mean()),
            }
            for name, frame in datasets.items()
        ]
    )
)

## 2. 滚动调参、最终训练与测试

In [ ]:
result = run_neural_benchmark(MODEL_NAME)
m, p, tuning, selected, fits, explanations, out = result
display(selected)
display(tuning.sort_values(["scenario", "validation_MAE"]).round(4))
display(m.round(4))
display(fits.round(4))

## 3. 最终预测与输入敏感度

In [ ]:
fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(22, 4), constrained_layout=True)
for ax, scenario in zip(axes, SCENARIOS):
    group = p.loc[(p.scenario == scenario) & p.split.eq("test")]
    ax.plot(group.ds, group.y, "ko-", ms=3, label="Observed")
    ax.plot(group.ds, group.prediction, label=MODEL_NAME)
    ax.set(title=scenario, ylabel="Units")
    ax.tick_params(axis="x", rotation=20)
    ax.legend()
fig.savefig(out / "final_test_predictions.png", dpi=150)
plt.show()
plt.close(fig)
display(
    explanations.sort_values(["scenario", "importance"], ascending=[True, False]).round(
        4
    )
)

## 4. 导出与复现检查

In [ ]:
required = {
    "metrics.csv",
    "predictions.csv",
    "tuning_results.csv",
    "selected_parameters.csv",
    "fit_details.csv",
    "explanations.csv",
    "manifest.json",
}
assert required.issubset({path.name for path in out.iterdir()})
assert set(m.scenario) == set(SCENARIOS)
assert set(m.split) == {"train_hindcast", "validation", "test"}
for name, digest in INPUT_HASHES.items():
    assert hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() == digest
print("Exports:", out.resolve())